[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/02_radiomap_and_mdt.ipynb)

# 02 — Reference Radio Data and Synthetic MDT

**Purpose.** Solve the Sionna-RT radio map at the deployed tilt, evaluate RSRP
along the UE trajectories from notebook 01 to build synthetic MDT records, and
compute the five KPIs. PROJECT.md section 16, Phase 3.

This notebook produces the **baseline** — the reference every reported
improvement in notebooks 05a, 05b and 06 is measured against. It is also where
the coordinate convention, the grid geometry and the KPI code first meet, so it
is where a mismatch between them is cheapest to find.

**Inputs.** The scenario from notebook 00, the trajectories from notebook 01,
`configs/radio.yaml`.

**Outputs.** The reference radio map `R`, the synthetic MDT records, the UE
density grid, and the baseline KPI vector.

**Requires** `uv sync --extra rt`.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook except the COLAB_PACKAGES line below, which names
# the extras this particular notebook needs. Forked the repository? Change these
# three values and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna_rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/external/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the scenario

The cell-band table and its row order come from notebook 00 and are not rebuilt
here — every tilt vector in the project is indexed by that order, so it has
exactly one producer.

The scene is loaded once. Building the acceleration structure over 3,753 meshes
costs far more than a single solve, and notebook 03 reuses this same scene for
hundreds of configurations.

In [ ]:
from src.data.load import load_cell_config
from src.radio import cell_band, radiomap, scene

cells_df = load_cell_config(cfg)
table = cell_band.build_table(cells_df, cfg)
tilt_0 = cell_band.current_tilt(table)

sc = scene.load_scene(cfg)
xmin, ymin, xmax, ymax = scene.scene_bounds(cfg)
print(f"{len(table)} cell-band pairs; scene x [{xmin}, {xmax}]  y [{ymin}, {ymax}]")

## 3. Coordinate and angle sanity check

**Do this before spending a solve.** A positive downtilt must produce a
*negative* pitch.

A sign error here does not raise. It produces a complete, plausible radio map
with every beam pointing at the sky, and an optimizer that converges confidently
on the wrong answer.

The PROJECT.md rewrite **removed** the coordinate-and-angle-conventions section,
so `src/radio/geometry.py` is now the only statement of this convention anywhere
in the project. That makes this check more load-bearing than it was, not less.

In [ ]:
from src.radio import geometry

orient = geometry.orientations(table, tilt_0)
check = pd.DataFrame(
    {
        "azimuth_deg": table["azimuth"],
        "tilt_deg": tilt_0,
        "yaw_rad": orient[:, 0],
        "pitch_rad": orient[:, 1],
    }
)
assert (check.loc[check.tilt_deg > 0, "pitch_rad"] < 0).all(), "downtilt must give negative pitch"
check.head()

## 4. Attach transmitters and solve at the deployed tilt

One transmitter per cell-band, in table order, so the returned radio map can be
mapped back to the table without guessing.

`R` is the common intermediate representation (PROJECT.md section 10): the KPI
evaluator, the surrogate's training target and the final validation all consume
this same array shape, which is what lets one KPI implementation serve all
three.

In [ ]:
sc = scene.add_transmitters(sc, table, cfg)
rsrp = radiomap.evaluate(tilt_0, sc, table, cfg)
print(f"RSRP array R: {rsrp.shape}  (cell-bands x grid cells)")
print(f"finite fraction: {np.isfinite(rsrp).mean():.1%}")

## 5. Synthetic MDT along the UE trajectories

PROJECT.md sections 9.2 and 9.3. The radio map above is a static grid; MDT is
what the network would have *measured*, at the positions UEs actually traversed
and at the times they were there.

The minimal record is `ue_id, sim_x, sim_y, timestamp, cell_id, band_id, rsrp,
ue_height`. Where several cell-band measurements share a record, `rsrp` becomes
a vector indexed by `(cell, band)`.

This is the part that distinguishes MDT from a grid sample: it carries the UE
distribution, and that distribution is what weights the Band Priority Score.

In [ ]:
# TODO(1): load the trajectories written by notebook 01, for this scenario_id
# TODO(2): evaluate RSRP at each (x, y, t) — per cell-band, at cfg.radio.grid.height_m
# TODO(3): assemble records with the section 9.3 columns, carrying scenario_id
# TODO(4): write to data/interim/, partitioned by scenario_id
raise NotImplementedError("synthetic MDT generation — needs src/mobility/ from notebook 01")

## 6. UE density on the same grid

`src.data.ue_density.build_grid` owns the grid geometry and nothing else may
define it. The alignment assertion is not ceremony: the two arrays broadcast
happily at mismatched offsets, and the resulting Band Priority Score is
plausible and wrong.

Built from the **training** scenarios only. Density built from all of them would
let held-out measurements shape the objective the optimizer maximises.

In [ ]:
from src.data.load import load_processed
from src.data.ue_density import assert_grid_aligned, ue_density

train_df = load_processed(cfg, "train")
rho = ue_density(train_df, cfg)
assert_grid_aligned(rho, rsrp)

print(f"N_UE = {rho.sum():,.0f} over {len(rho):,} grid cells")
print(f"grid cells with no observations: {(rho == 0).mean():.1%}")

## 7. The baseline KPI vector

The reference for the whole project. Every improvement claimed in notebooks 05a,
05b and 06 is stated relative to these five numbers.

They are listed in `cfg.kpi.order`, which is the lexicographic priority:
hole rate, overlap rate, mean overlap neighbours, Band Priority Score, weak
rate. Weak rate is **last** — it moved there in the PROJECT.md rewrite, so a
configuration may now trade weak coverage for band coordination.

In [ ]:
from src.kpi import vector

baseline_kpis = vector.kpi_vector(rsrp, rho, table, cfg)
pd.Series(baseline_kpis).to_frame("baseline").loc[list(cfg.kpi.order)]

## 8. Internal consistency of the simulation

Under the previous formulation this section compared simulated RSRP against a
real operator MDT export. That export is retired: MDT is now generated by this
same simulator (PROJECT.md sections 6 and 9), so comparing the two would compare
the simulation with itself and prove nothing.

What can still be checked is internal consistency and physical plausibility —
and, in notebook 06, robustness across perturbed scenarios, which is what
section 12 replaces the reality check with.

In [ ]:
# TODO(1): RSRP at the MDT positions must match the radio map at the same grid
#          cell, to within the grid resolution — a mismatch is an indexing bug
# TODO(2): path loss vs. distance per band — the higher band must fall off faster
# TODO(3): serving-cell assignment vs. azimuth — cells should serve their own
#          sector, and a cell serving behind itself is a geometry error
# TODO(4): record the acceptance threshold before looking at the result

## 9. Spatial maps

The visual counterpart of section 7. A KPI table says hole rate is 8%; these say
whether the holes are at the edge of the area or in the middle of the densest UE
cluster, which is a very different outcome for the same number.

These are the maps PROJECT.md section 16 Phase 8 asks for in the final report.

In [ ]:
from src.evaluation import analysis
from src.kpi import coverage, serving

per_cell = serving.cell_rsrp(rsrp, table, cfg)
n_ov = coverage.overlap_neighbors(per_cell, serving.serving_cell(per_cell), cfg)
b_star = serving.dominant_band(rsrp, table)

analysis.rsrp_map(rsrp, cfg, title="Baseline RSRP")
analysis.coverage_map(rsrp, cfg)
analysis.overlap_map(n_ov, cfg)
analysis.ue_density_map(rho, cfg)
analysis.dominant_band_map(b_star, table, cfg)

## 10. Handoff checklist

- [ ] The angle check in section 3 passed: positive downtilt gives negative pitch.
- [ ] Cells and UE positions both sit inside the scene extent.
- [ ] `cfg.radio.grid.cell_size_m` is fixed and recorded — changing it later
      invalidates every surrogate sample generated against it.
- [ ] `cfg.radio.ray_tracing` is fixed and recorded, for the same reason.
- [ ] The baseline KPI vector is saved to `reports/results/`, with `scenario_id`.
- [ ] Synthetic MDT carries `scenario_id` on every record, so notebook 03 can
      split on it.
- [ ] The consistency checks in section 8 were run and accepted, or the
      discrepancy is recorded as a known limitation.

**Blocking gap.** Section 5 needs the trajectories from notebook 01, and
`src/mobility/` does not exist yet.